Join Types

In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql import Row

In [0]:
# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='New York'),
    Row(employee_id=2, name='Bob', city='Los Angeles'),
    Row(employee_id=3, name='Charlie', city='Chicago'),
    Row(employee_id=4, name='David', city='Houston'),
    Row(employee_id=5, name='Eva', city='Phoenix')
]

# Sample Salary info
data_salary = [
    Row(employee_id=1, salary=70000, deprt='HR'),
    Row(employee_id=2, salary=80000, deprt='Engineering'),
    Row(employee_id=3, salary=65000, deprt='Finance'),
    Row(employee_id=6, salary=90000, deprt='Marketing'),
    Row(employee_id=7, salary=75000, deprt='Sales')
]

spark = SparkSession.getActiveSession()
df_employee = spark.createDataFrame(data_employee)
df_salary = spark.createDataFrame(data_salary)

In [0]:
# Register temp views for SQL

# Employee info
df_employee.createOrReplaceTempView('employee')
# Salary info
df_salary.createOrReplaceTempView('salary')

display(df_employee)
display(df_salary)

employee_id,name,city
1,Alice,New York
2,Bob,Los Angeles
3,Charlie,Chicago
4,David,Houston
5,Eva,Phoenix


employee_id,salary,deprt
1,70000,HR
2,80000,Engineering
3,65000,Finance
6,90000,Marketing
7,75000,Sales


Inner Join

In [0]:
df_inner = df_employee.join(df_salary, on = 'employee_id',how='inner')
display(df_inner)

employee_id,name,city,salary,deprt
1,Alice,New York,70000,HR
2,Bob,Los Angeles,80000,Engineering
3,Charlie,Chicago,65000,Finance


Outer Join

In [0]:
df_outer = df_employee.join(df_salary,on='employee_id',how='outer')
display(df_outer)

employee_id,name,city,salary,deprt
1,Alice,New York,70000,HR
2,Bob,Los Angeles,80000,Engineering
3,Charlie,Chicago,65000,Finance
4,David,Houston,null,null
5,Eva,Phoenix,null,null
7,null,null,75000,Sales
6,null,null,90000,Marketing


Left Outer Join

In [0]:
df_left_outer = df_employee.join(df_salary,on='employee_id',how='left_outer')
display(df_left_outer)

employee_id,name,city,salary,deprt
1,Alice,New York,70000,HR
2,Bob,Los Angeles,80000,Engineering
3,Charlie,Chicago,65000,Finance
4,David,Houston,null,null
5,Eva,Phoenix,null,null


Right Outer Join

In [0]:
df_right_outer = df_employee.join(df_salary,on='employee_id',how='right_outer')
display(df_right_outer)

employee_id,name,city,salary,deprt
1,Alice,New York,70000,HR
2,Bob,Los Angeles,80000,Engineering
3,Charlie,Chicago,65000,Finance
6,null,null,90000,Marketing
7,null,null,75000,Sales


Left Semi Join

A left semi join returns only the rows from the left dataframe (Employee info) where the join key exists in the right dataframe (Salary info). It does not include columns from the right dataframe.

In [0]:
df_left_semi = df_employee.join(df_salary,on="employee_id",how='left_semi')
display(df_left_semi)

employee_id,name,city
1,Alice,New York
2,Bob,Los Angeles
3,Charlie,Chicago


Left Anti Join

A left anti join returns only the rows from the left dataframe (Employee info) where the join key does NOT exist in the right dataframe (Salary info). It does not include columns from the right dataframe.

In [0]:
df_left_anti = df_employee.join(df_salary,on='employee_id',how='left_anti')
display(df_left_anti)

employee_id,name,city
4,David,Houston
5,Eva,Phoenix


Cross Join

A cross join returns the Cartesian product of the two dataframes, matching every row in the left dataframe with every row in the right dataframe. Use with caution as it can produce a large number of rows.


In [0]:
df_cross = df_employee.crossJoin(df_salary)
display(df_cross)

employee_id,name,city,employee_id,salary,deprt
1,Alice,New York,1,70000,HR
1,Alice,New York,2,80000,Engineering
1,Alice,New York,3,65000,Finance
1,Alice,New York,6,90000,Marketing
1,Alice,New York,7,75000,Sales
2,Bob,Los Angeles,1,70000,HR
2,Bob,Los Angeles,2,80000,Engineering
2,Bob,Los Angeles,3,65000,Finance
2,Bob,Los Angeles,6,90000,Marketing
2,Bob,Los Angeles,7,75000,Sales


*Handling Duplicates Rows and Same Column Names in Joins*

When joining dataframes, duplicate rows can be removed using `dropDuplicates()`. If both dataframes have columns with the same name (other than join keys), PySpark automatically adds suffixes to distinguish them. We can also use `select` or `withColumnRenamed` to resolve column name conflicts.

In [0]:
# Add a duplicate row to df_employee for demonstration
df_employee_dup = df_employee.union(df_employee.filter(df_employee.employee_id == 1))
# Remove duplicates
df_employee_nodup = df_employee_dup.dropDuplicates()
# Join and resolve column name conflicts
joined = df_employee_nodup.join(df_salary, on='employee_id', how='inner')
# Rename columns if needed
joined = joined.withColumnRenamed('name', 'employee_name')
display(joined)

employee_id,employee_name,city,salary,deprt
1,Alice,New York,70000,HR
2,Bob,Los Angeles,80000,Engineering
3,Charlie,Chicago,65000,Finance


**Join Strategies**

*Broadcast Join*

A broadcast join is a Spark optimization technique that copies a small dataset to every worker node (executor) so that a large dataset can be joined locally without network shuffling.

*Hash Join*

A hash join is a relational database algorithm used to combine rows from two tables efficiently based on an equality condition.

*Shuffle Hash Join*

A Shuffle Hash Join is an Apache Spark join strategy that shuffles both datasets across the cluster by their join key and builds an in-memory hash table for each partition of the smaller dataset to probe against the larger dataset.

In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql import Row

# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='New York'),
    Row(employee_id=2, name='Bob', city='Los Angeles'),
    Row(employee_id=3, name='Charlie', city='Chicago'),
    Row(employee_id=4, name='David', city='Houston'),
    Row(employee_id=5, name='Eva', city='Phoenix')
]

# Sample Salary info
data_salary = [
    Row(employee_id=1, salary=70000, deprt='HR'),
    Row(employee_id=2, salary=80000, deprt='Engineering'),
    Row(employee_id=3, salary=65000, deprt='Finance'),
    Row(employee_id=6, salary=90000, deprt='Marketing'),
    Row(employee_id=7, salary=75000, deprt='Sales')
]

spark = SparkSession.getActiveSession()
df_employee = spark.createDataFrame(data_employee)
df_salary = spark.createDataFrame(data_salary)

# Register temp views for SQL

# Employee info
df_employee.createOrReplaceTempView('employee')
# Salary info
df_salary.createOrReplaceTempView('salary')

display(df_employee)
display(df_salary)

employee_id,name,city
1,Alice,New York
2,Bob,Los Angeles
3,Charlie,Chicago
4,David,Houston
5,Eva,Phoenix


employee_id,salary,deprt
1,70000,HR
2,80000,Engineering
3,65000,Finance
6,90000,Marketing
7,75000,Sales


In [0]:
#Broadcast Join Threshold

try:
    threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold") #default = 10MB
except Exception:
    threshold = "10485760 (10MB) — not accessible on Serverless compute"
print(threshold)

10485760 (10MB) — not accessible on Serverless compute


In [0]:
#Setting Broadcast join threshold

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1gb") #10MB
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(threshold)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6126443614551881>, line 3
      1 #Setting Broadcast join threshold
----> 3 spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1gb") #10MB
      4 threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
      5 print(threshold)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:48, in RuntimeConf.set(self, key, value)
     46 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     47 operation = proto.ConfigRequest.Operation(set=op_set)
---> 48 result = self._client.config(operation)
     49 for warn in result.warnings:
     50     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2525, in SparkConnectClient.config(self, operation)
   2523     raise SparkConnectException("Invalid state durin

In [0]:
#Getting Physical Plan
df_employee.join(df_salary, on='employee_id', how='inner').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [employee_id#10230L, name#10231, city#10232, salary#10249L, deprt#10250]
         +- PhotonBroadcastHashJoin [employee_id#10230L], [employee_id#10248L], Inner, BuildRight, false, true, false
            :- PhotonRowToColumnar
            :  +- LocalTableScan [employee_id#10230L, name#10231, city#10232]
            +- PhotonShuffleExchangeSource false
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#7653]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [employee_id#10248L, salary#10249L, deprt#10250]


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
#Hint for broadcast join

df=df_employee.join(df_salary.hint("broadcast"), on='employee_id', how='inner')
display(df)

employee_id,name,city,salary,deprt
1,Alice,New York,70000,HR
2,Bob,Los Angeles,80000,Engineering
3,Charlie,Chicago,65000,Finance
